In [96]:
import os
import json
import openai
import re
import importlib
import matplotlib.pyplot as plt
from openai import OpenAI

In [97]:
# Load the API key into the client for OpenAI API access.
def load_api_key(file_path='api_key.txt'):
    with open(file_path, 'r') as f:
        for line in f:
            if line.startswith('api_key='):
                return line.strip().split('=', 1)[1]
    return None

api_key = load_api_key()

if api_key is None:
    print("Error: API key not found.")
    exit()

client = OpenAI(api_key=api_key,)

chat_completion = client.chat.completions.create(
    messages=[
        {
            "role": "user",
            "content": "Say this is a test",
        }
    ],
    model="gpt-4o-mini",
)

In [98]:
# Load prompts from the file
def load_prompts_from_file(file_path):
    with open(file_path, 'r', encoding='utf-8') as f:
        content = f.read()  # Read the entire file content

    # Split the content into system and user sections using markers
    system_prompt = content.split('--USER--')[0].replace('--SYSTEM--', '').strip()
    user_prompt = content.split('--USER--')[1].strip()
    
    return system_prompt, user_prompt

In [99]:
def personality_generation(num, output_dir, output_file, prompt_path, client):
    # Ensure output directory exists
    os.makedirs(output_dir, exist_ok=True)
    
    output_path = os.path.join(output_dir, output_file)
    
    print(f"[AgentSense] Generating {num} personalities...")
    
    # Load system and user prompts from file
    system_prompt, user_prompt = load_prompts_from_file(prompt_path)
    
    # Prepare the user content with the number of people to generate
    prompt = f'Generate descriptions for {num} people.'
    
    # Generate the response from the GPT model
    response = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=[
            {"role": "system", "content": system_prompt},
            {"role": "user", "content": prompt + ' ' + user_prompt}
        ]
    )

    generated_text = response.choices[0].message.content

    # Save to file
    with open(output_path, "w", encoding="utf-8") as f:
        f.write(generated_text)
        
    print(f"[AgentSense] Personality generation finished. Output saved to: {output_path.replace(os.sep, '/')}")
    
    return generated_text

In [100]:
# =========================
# Configuration (User-editable)
# =========================

# Path to prompt file
PROMPT_PATH = "prompts/personality_generation_prompt.txt"

# Directory to store generated personality data
DATA_DIR = "data/step_1_personality_data"

# Output file name
OUTPUT_FILE = "personality_generated.txt"

# Number of personalities to generate per run
# Recommended: at most 5 to control API usage
NUM_PERSONALITIES = 5

# =========================
# Run 
# =========================

personality_generation(
    num=NUM_PERSONALITIES,
    output_dir=DATA_DIR,
    output_file=OUTPUT_FILE,
    prompt_path=PROMPT_PATH,
    client=client
);

[AgentSense] Generating 5 personalities...
[AgentSense] Personality generation finished. Output saved to: data/personality_data/personality_generated.txt
